In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

#ADLS configuration 

# ADLS configuration
storage_key = dbutils.secrets.get(
    scope="hospitalanalyticsvaultscope",
    key="storage-conn"
).strip()

spark.conf.set(
  "fs.azure.account.key.hospitalstorageadls.dfs.core.windows.net",
  storage_key
)

bronze_path = "abfss://bronze@hospitalstorageadls.dfs.core.windows.net/patient_flow"
silver_path = "abfss://silver@hospitalstorageadls.dfs.core.windows.net/patient_flow"

#read from bronze
bronze_df = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
)

#Defin Schema
schema = StructType([
    StructField("patient_id", StringType()),
    StructField("gender", StringType()),
    StructField("age", IntegerType()),
    StructField("department", StringType()),
    StructField("admission_time", StringType()),
    StructField("discharge_time", StringType()),
    StructField("bed_id", IntegerType()),
    StructField("hospital_id", IntegerType())
])

#Parse it to dataframe
parsed_df = bronze_df.withColumn("data",from_json(col("raw_json"),schema)).select("data.*")

#convert type to Timestamp
clean_df = parsed_df.withColumn("admission_time", to_timestamp("admission_time"))
clean_df = clean_df.withColumn("discharge_time", to_timestamp("discharge_time"))

#invalid admission_times
clean_df = clean_df.withColumn("admission_time",
                               when(
                                   col("admission_time").isNull() | (col("admission_time") > current_timestamp()),
                                   current_timestamp())
                               .otherwise(col("admission_time")))

#Handle Invalid Age
clean_df = clean_df.withColumn("age",
                               when(col("age")>100,floor(rand()*90+1).cast("int"))
                               .otherwise(col("age"))
                               )

#schema evolution
expected_cols = ["patient_id", "gender", "age", "department", "admission_time", "discharge_time", "bed_id", "hospital_id"]

for col_name in expected_cols:
    if col_name not in clean_df.columns:
        clean_df = clean_df.withColumn(col_name, lit(None))

#Write to silver table
(
    clean_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("mergeSchema","true")
    .option("checkpointLocation", silver_path + "_checkpoint")
    .start(silver_path)
)



In [0]:
display(spark.read.format("delta").load(silver_path))

patient_id,gender,age,department,admission_time,discharge_time,bed_id,hospital_id
d8f37a6e-bda1-49e8-9ff1-90aa23ac1bdb,Male,98,Emergency,2026-11-20T03:19:12.105655Z,2026-11-20T04:19:12.105655Z,321,1
edf342d1-7f9f-4e80-b740-bb1a4632d43d,Female,96,Surgery,2026-11-21T05:19:13.280807Z,2026-11-21T22:19:13.280807Z,249,1
da003da3-4de4-4dbc-b22e-1f91c4566fed,Female,61,ICU,2026-11-21T13:19:14.283449Z,2026-11-21T21:19:14.283449Z,457,1
49b1537d-025e-44d1-b492-2650530b9031,Male,25,Oncology,2026-11-20T10:19:15.285283Z,2026-11-22T12:19:15.285283Z,55,5
ac732a3e-8f87-471b-9c57-c760afb65d38,Female,24,Surgery,2026-11-19T09:19:16.28718Z,2026-11-22T03:19:16.28718Z,226,3
e37635b9-f983-481a-9674-fe56d1f94c84,Male,2,Surgery,2026-11-22T07:19:17.290139Z,2026-11-23T20:19:17.290139Z,331,1
50e6273d-1518-4a2a-9267-26a139d1415b,Female,53,Emergency,2026-11-21T04:19:18.412365Z,2026-11-23T00:19:18.412365Z,395,3
5e5f04dd-af8a-4b6a-84df-684409a31208,Female,3,Pediatrics,2026-11-19T15:19:19.416112Z,2026-11-20T07:19:19.416112Z,493,2
b8671fb6-62e9-447a-bb71-ac52b100902c,Male,64,Oncology,2026-11-19T19:19:20.419575Z,2026-11-20T08:19:20.419575Z,395,5
258d444b-98d6-4a83-a7c5-204e2ec5fc06,Female,26,Maternity,2026-11-21T05:19:21.420893Z,2026-11-23T15:19:21.420893Z,188,2
